In [1]:
import numpy as np
print("All modules imported successfully!")

All modules imported successfully!


In [ ]:
# The most basic RNN implementation from scratch in Python using NumPy. This implementation includes the forward pass, backward pass, and parameter update steps for training the RNN on a sequence of data.

class RNN:
    def __init__(self, input_size, hidden_size, output_size):
        self.hs = hidden_size
        self.is_ = input_size
        self.os = output_size

        self.Whh = np.random.randn(self.hs, self.hs) * 0.01
        self.Wxh = np.random.randn(self.hs, self.is_) * 0.01
        self.Why = np.random.randn(self.os, self.hs) * 0.01

        self.bh = np.zeros((self.hs, 1))
        self.by = np.zeros((self.os, 1))

    def forward(self, X):
        """X shape : (sequence_length, input_size, 1)"""
        self.X_history = X
        h_prev = np.zeros((self.hs, 1))
        self.h_history = [h_prev]
        self.y_history = []

        for t in range(len(X)):
            xt = X[t]
            zt = np.dot(self.Wxh, xt) + np.dot(self.Whh, h_prev) + self.bh
            ht = np.tanh(zt)

            yt = np.dot(self.Why, ht) + self.by
            self.h_history.append(ht)
            self.y_history.append(yt)
            h_prev = ht
        return self.y_history

    def backward(self, dy_history):
        dWhh = np.zeros_like(self.Whh)
        dWhy = np.zeros_like(self.Why)
        dWxh = np.zeros_like(self.Wxh)
        dbh = np.zeros_like(self.bh)

        dh_next = np.zeros((self.hs, 1))
        seq_len = len(self.X_history)

        for t in reversed(range(seq_len)):
            xt = self.X_history[t]
            ht = self.h_history[t+1]
            h_prev = self.h_history[t]

            dyt = dy_history[t]
            dWhy += np.dot(dyt, ht.T)

            dht = np.dot(self.Why.T, dyt) + dh_next
            dzt = dht * (1 - ht ** 2)

            dWhh += np.dot(dzt, h_prev.T)
            dWxh += np.dot(dzt, xt.T)
            dbh += dzt

            dh_next = np.dot(self.Whh.T, dzt)

        return dWxh, dWhh, dWhy, dbh

    def update_parameters(self, dWxh, dWhh, dWhy, dbh, learning_rate):
        self.Wxh -= learning_rate * dWxh
        self.Whh -= learning_rate * dWhh
        self.Why -= learning_rate * dWhy
        self.bh -= learning_rate * dbh
        return

    def train(self, X, Y, learning_rate=0.01):
        y_pred = self.forward(X)
        dy_history = [y_pred[t] - Y[t] for t in range(len(Y))]
        dWxh, dWhh, dWhy, dbh = self.backward(dy_history)
        self.update_parameters(dWxh, dWhh, dWhy, dbh, learning_rate)
        return y_pred
    




In [2]:
class BatchedRNNWithEmbedding:
    def __init__(self, vocab_size, embed_size, hidden_size, output_size):
        self.vs = vocab_size
        self.es = embed_size
        self.hs = hidden_size
        self.os = output_size
        
        # 1. Embedding Layer: Shape (vocab_size x embed_size)
        # We look up rows in this matrix to get dense vectors for words
        self.We = np.random.randn(self.vs, self.es) * 0.1
        
        # 2. RNN Core Weights
        # Whh: (m x m), Wxh: (m x d), Why: (k x m)
        self.Whh = np.random.randn(self.hs, self.hs) * 0.1
        self.Wxh = np.random.randn(self.hs, self.es) * 0.1
        self.Why = np.random.randn(self.os, self.hs) * 0.1
        
        # Biases are column vectors
        self.bh = np.zeros((self.hs, 1))
        self.by = np.zeros((self.os, 1))

    def forward(self, X):
        """
        X shape: (sequence_length, batch_size) -> contains integer token IDs
        """
        self.seq_len, self.batch_size = X.shape
        self.X_history = X
        
        # h_prev shape: (hidden_size x batch_size) -> (m x B)
        h_prev = np.zeros((self.hs, self.batch_size))
        self.h_history = [h_prev]
        self.y_history = []
        self.embed_history = []
        
        for t in range(self.seq_len):
            # 1. Get batch of word IDs at time t
            word_ids = X[t] 
            
            # 2. Embedding Lookup 
            # embedded_xt shape: (batch_size, embed_size)
            embedded_xt = self.We[word_ids] 
            
            # Transpose to match our RNN column-vector math -> (embed_size x batch_size)
            xt_transposed = embedded_xt.T 
            self.embed_history.append(xt_transposed)
            
            # 3. RNN Forward Step
            # (m x m)@(m x B) + (m x d)@(d x B) + (m x 1 broadcasted to m x B)
            zt = np.dot(self.Whh, h_prev) + np.dot(self.Wxh, xt_transposed) + self.bh
            ht = np.tanh(zt)
            
            # 4. Output Step (Logits)
            # (k x m)@(m x B) + (k x 1) -> (k x B)
            yt = np.dot(self.Why, ht) + self.by
            
            self.h_history.append(ht)
            self.y_history.append(yt)
            h_prev = ht
            
        return self.y_history

    def backward(self, dy_history):
        """
        dy_history contains the gradients of the loss w.r.t the outputs.
        List of length seq_len, each item is shape (output_size x batch_size)
        """
        dWhh = np.zeros_like(self.Whh)
        dWxh = np.zeros_like(self.Wxh)
        dWhy = np.zeros_like(self.Why)
        dbh = np.zeros_like(self.bh)
        dby = np.zeros_like(self.by)
        dWe = np.zeros_like(self.We)
        
        # Future error initially zero
        dh_next = np.zeros((self.hs, self.batch_size))
        
        for t in reversed(range(self.seq_len)):
            # Retrieve historical states
            word_ids = self.X_history[t]
            xt = self.embed_history[t] # Shape: (embed_size x batch_size)
            ht = self.h_history[t+1]
            h_prev = self.h_history[t]
            
            # 1. Output Gradients
            dyt = dy_history[t] # Shape: (output_size x batch_size)
            dWhy += np.dot(dyt, ht.T)
            
            # CRITICAL BATCHING STEP: Sum across batch (axis=1) to collapse back to (os x 1)
            dby += np.sum(dyt, axis=1, keepdims=True)
            
            # 2. Hidden State Error
            dht = np.dot(self.Why.T, dyt) + dh_next
            dzt = dht * (1 - ht ** 2)
            
            # 3. Recurrent Weight Gradients
            dWhh += np.dot(dzt, h_prev.T)
            dWxh += np.dot(dzt, xt.T)
            
            # CRITICAL BATCHING STEP: Sum across batch
            dbh += np.sum(dzt, axis=1, keepdims=True)
            
            dh_next = np.dot(self.Whh.T, dzt)
            
            # 4. Embedding Layer Gradient
            # Pass error back through Wxh to the embedded inputs
            dxt = np.dot(self.Wxh.T, dzt) # Shape: (embed_size x batch_size)
            
            # Route gradients to the specific word vectors used in this batch
            # We iterate through the batch to update the correct rows in the embedding matrix
            for b in range(self.batch_size):
                token_id = word_ids[b]
                dWe[token_id] += dxt[:, b]
                
        return dWe, dWxh, dWhh, dWhy, dbh, dby

    def update_parameters(self, grads, lr):
        dWe, dWxh, dWhh, dWhy, dbh, dby = grads
        # Simple SGD update
        self.We -= lr * dWe
        self.Wxh -= lr * dWxh
        self.Whh -= lr * dWhh
        self.Why -= lr * dWhy
        self.bh -= lr * dbh
        self.by -= lr * dby

In [3]:
def softmax(x):
    # Subtract max for numerical stability
    e_x = np.exp(x - np.max(x, axis=0, keepdims=True))
    return e_x / e_x.sum(axis=0, keepdims=True)

# 1. Dataset Preparation
# We map characters/words to integers
vocab = {"<pad>": 0, "ser": 1, "duncan": 2, "the": 3, "tall": 4, "was": 5, "a": 6, "hedge": 7, "knight": 8}
reverse_vocab = {v: k for k, v in vocab.items()}
vocab_size = len(vocab)

# Let's train on two sequences simultaneously (Batch Size = 2)
# Seq 1: "ser duncan the tall"
# Seq 2: "was a hedge knight"
X_train = np.array([
    [1, 5], # t=0: [ser, was]
    [2, 6], # t=1: [duncan, a]
    [3, 7], # t=2: [the, hedge]
]) # Shape: (Sequence Length=3, Batch Size=2)

# Targets are the next word in the sequence
Y_train = np.array([
    [2, 6], # predict: [duncan, a]
    [3, 7], # predict: [the, hedge]
    [4, 8], # predict: [tall, knight]
])

# 2. Model Initialization
embed_size = 8
hidden_size = 16
output_size = vocab_size # Predicting a probability distribution over the whole vocab
lr = 0.1

model = BatchedRNNWithEmbedding(vocab_size, embed_size, hidden_size, output_size)

# 3. Training Loop
epochs = 200
for epoch in range(epochs):
    # --- Forward Pass ---
    logits_history = model.forward(X_train)
    
    loss = 0
    dy_history = []
    
    # Calculate loss and gradients at each time step
    for t in range(model.seq_len):
        logits = logits_history[t] # Shape: (vocab_size x batch_size)
        targets = Y_train[t]       # Shape: (batch_size,)
        
        # Compute Softmax Probabilities
        probs = softmax(logits)
        
        # Compute Cross Entropy Loss for the batch
        # We index the probabilities of the correct target words
        batch_loss = -np.log(probs[targets, range(model.batch_size)])
        loss += np.sum(batch_loss) / model.batch_size
        
        # Derivative of Softmax + CrossEntropy is simply: Probs - True_Labels (One-Hot)
        dy = probs.copy()
        dy[targets, range(model.batch_size)] -= 1
        # Scale the gradient by batch size so learning rate stays stable
        dy /= model.batch_size 
        
        dy_history.append(dy)
        
    # --- Backward Pass ---
    grads = model.backward(dy_history)
    model.update_parameters(grads, lr)
    
    if epoch % 50 == 0:
        print(f"Epoch {epoch} | Loss: {loss:.4f}")

# 4. Inference Test
print("\n--- Testing Predictions ---")
final_logits = model.forward(X_train)
for t in range(model.seq_len):
    probs = softmax(final_logits[t])
    # Get the index with the highest probability for each sequence in the batch
    predictions = np.argmax(probs, axis=0)
    
    word_seq1 = reverse_vocab[predictions[0]]
    word_seq2 = reverse_vocab[predictions[1]]
    print(f"Time Step {t+1} predictions -> Seq 1: '{word_seq1}' | Seq 2: '{word_seq2}'")

Epoch 0 | Loss: 6.5742
Epoch 50 | Loss: 1.0599
Epoch 100 | Loss: 0.1247
Epoch 150 | Loss: 0.0591

--- Testing Predictions ---
Time Step 1 predictions -> Seq 1: 'duncan' | Seq 2: 'a'
Time Step 2 predictions -> Seq 1: 'the' | Seq 2: 'hedge'
Time Step 3 predictions -> Seq 1: 'tall' | Seq 2: 'knight'
